<a href="https://colab.research.google.com/github/shovo896/OpenCV/blob/main/train_sleep_disorder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model


# Load Dataset

df = pd.read_csv("/content/Sleep_health_and_lifestyle_dataset.csv")

# Features (without BMI)
features = ["Stress Level", "Physical Activity Level", "Age"]
target = "Quality of Sleep"

X = df[features].values
y = df[target].values

# Normalize
X = (X - X.mean(axis=0)) / X.std(axis=0)
y = (y - y.mean()) / y.std()


# Define PINN model

class PINN(Model):
    def __init__(self):
        super(PINN, self).__init__()
        self.hidden1 = layers.Dense(32, activation="tanh")
        self.hidden2 = layers.Dense(32, activation="tanh")
        self.out = layers.Dense(1)

        # Physics parameters (a, b, c)
        self.a = tf.Variable(0.1, trainable=True, dtype=tf.float32)
        self.b = tf.Variable(0.1, trainable=True, dtype=tf.float32)
        self.c = tf.Variable(0.1, trainable=True, dtype=tf.float32)

    def call(self, x):
        h = self.hidden1(x)
        h = self.hidden2(h)
        return self.out(h)


# Loss Function: Data + Physics

def pinn_loss(model, x, y_true):
    with tf.GradientTape() as tape:
        tape.watch(x)
        y_pred = model(x)

    # Compute gradient wrt inputs
    dy_dx = tape.gradient(y_pred, x)

    # Physics residual (ODE)
    stress, activity, age = tf.unstack(x, axis=1)
    physics_residual = dy_dx[:,0] + model.a*stress - model.b*activity + model.c*age

    # Total loss
    data_loss = tf.reduce_mean(tf.square(y_true - y_pred[:,0]))
    physics_loss = tf.reduce_mean(tf.square(physics_residual))
    return data_loss + physics_loss


# Training Loop

model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

X_tf = tf.convert_to_tensor(X, dtype=tf.float32)
y_tf = tf.convert_to_tensor(y, dtype=tf.float32)

for epoch in range(500):
    with tf.GradientTape() as tape:
        loss_value = pinn_loss(model, X_tf, y_tf)
    grads = tape.gradient(loss_value, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss_value.numpy():.4f}")


# Predictions

preds = model(X_tf).numpy()
print("Sample Predictions:", preds[:5])


Epoch 0, Loss: 0.8997
Epoch 50, Loss: 0.4814
Epoch 100, Loss: 0.4076
Epoch 150, Loss: 0.2967
Epoch 200, Loss: 0.2406
Epoch 250, Loss: 0.1994
Epoch 300, Loss: 0.1556
Epoch 350, Loss: 0.1222
Epoch 400, Loss: 0.1042
Epoch 450, Loss: 0.0954
Sample Predictions: [[-1.2992208 ]
 [-0.78459436]
 [-0.78459436]
 [-1.6124982 ]
 [-1.6124982 ]]
